# Notebook Explore Analysis

Este notebook tiene como objetivo comprender la estructura, calidad y comportamiento inicial de los datos antes de construir el modelo de forecasting.

El análisis se enfoca en validar la granularidad de la información, revisar consistencia entre tablas, 
identificar valores faltantes y observar patrones generales de demanda que puedan orientar las decisiones de modelado.

## Librerias 

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

import seaborn as sns

from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

## Data

In [53]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

transactions = pd.read_csv(ROOT / "data" / "raw" / "transactions.csv")
stores = pd.read_csv(ROOT / "data" / "raw" / "stores.csv")
calendar = pd.read_csv(ROOT / "data" / "raw" / "calendar.csv")

## EDA

### Transactions

El dataset de transacciones constituye la principal fuente de información del proyecto, ya que concentra la demanda observada para cada combinación de tienda, categoría y día. Antes de construir un modelo de pronóstico es necesario validar que esta información sea consistente y suficientemente completa, pues cualquier anomalía podría propagarse al resto del proceso analítico.

In [3]:
transactions.head(2)

,date,store_id,category,total_transactions,cash_transactions,card_transactions,amount_total,amount_cash,amount_card,units_sold,avg_ticket,has_promotion,replenishment_signal
0,2023-01-01,STR_001,Abarrotes,833,NaN,326,244732.98,NaN,101822.79,2299.0,NaN,0,NaN
1,2023-01-02,STR_001,Abarrotes,1298,750.0,548,373235.09,222834.25,150400.84,3011.0,287.55,0,NaN


In [6]:
transactions.shape

(203958, 13)

In [7]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203958 entries, 0 to 203957
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   date                  203958 non-null  object 
 1   store_id              203958 non-null  object 
 2   category              203958 non-null  object 
 3   total_transactions    203958 non-null  int64  
 4   cash_transactions     191835 non-null  float64
 5   card_transactions     203958 non-null  int64  
 6   amount_total          203958 non-null  float64
 7   amount_cash           191835 non-null  float64
 8   amount_card           203958 non-null  float64
 9   units_sold            197840 non-null  float64
 10  avg_ticket            199084 non-null  float64
 11  has_promotion         203958 non-null  int64  
 12  replenishment_signal  202998 non-null  float64
dtypes: float64(7), int64(3), object(3)
memory usage: 20.2+ MB


In [42]:
null_counts = transactions.isnull().sum()
null_pct = (null_counts / len(transactions) * 100).round(2)

null_summary = pd.DataFrame({
    'Valores nulos': null_counts,
    '% del total': null_pct
}).query('`Valores nulos` > 0').sort_values('% del total', ascending=False)

null_summary

,Valores nulos,% del total
cash_transactions,12123,5.94
amount_cash,12123,5.94
units_sold,6118,3.00
avg_ticket,4874,2.39
replenishment_signal,960,0.47


In [44]:
transactions[transactions['cash_transactions'].isna()].amount_cash.isna().sum()

12123

cash_transactions y amount_cash (~6%): presentan exactamente el mismo patrón de valores faltantes, lo que resulta consistente al tratarse de métricas asociadas a pagos en efectivo y cuyo diccionario de datos indica que pueden contener nulos.


replenishment_signal (~0.5%): corresponde a una variable generada por un proceso interno del negocio. Debido a que se desconoce la lógica con la que fue construida, su tratamiento se evaluará durante la etapa de preparación de datos.


units_sold (~3%) y avg_ticket (~2.4%): presentan un porcentaje moderado de valores faltantes que deberá considerarse en la estrategia de preprocesamiento

In [45]:
transactions['date'] = pd.to_datetime(transactions['date'])

total_days = (transactions['date'].max() - transactions['date'].min()).days + 1
days_per_store = transactions.groupby('store_id')['date'].nunique()

incomplete_stores = days_per_store[days_per_store < total_days].reset_index()
incomplete_stores.columns = ['store_id', 'dias_registrados']
incomplete_stores['dias_faltantes'] = total_days - incomplete_stores['dias_registrados']
incomplete_stores['pct_faltante'] = (incomplete_stores['dias_faltantes'] / total_days * 100).round(2)

print(f"Días esperados por tienda: {total_days}")
print(f"Tiendas con días incompletos: {len(incomplete_stores)} de {transactions['store_id'].nunique()}\n")
incomplete_stores

Días esperados por tienda: 425
Tiendas con días incompletos: 3 de 80



,store_id,dias_registrados,dias_faltantes,pct_faltante
0,STR_045,423,2,0.47
1,STR_055,422,3,0.71
2,STR_064,423,2,0.47


In [47]:
# Verificar si los días faltantes son aislados o periodos consecutivos
for store in incomplete_stores['store_id']:
    store_dates = transactions[transactions['store_id'] == store]['date'].sort_values()
    all_dates = pd.date_range(start=transactions['date'].min(), end=transactions['date'].max())
    missing_dates = all_dates[~all_dates.isin(store_dates)]
    
    # Detectar si los gaps son consecutivos
    gaps = (missing_dates.to_series().diff() > pd.Timedelta('1 day')).sum()
    print(f"{store}: {len(missing_dates)} días faltantes — {'periodos consecutivos' if gaps < len(missing_dates) - 1 else 'días aislados'} ({gaps + 1} bloque(s) de ausencia)")

STR_045: 2 días faltantes — periodos consecutivos (1 bloque(s) de ausencia)
STR_055: 3 días faltantes — periodos consecutivos (1 bloque(s) de ausencia)
STR_064: 2 días faltantes — periodos consecutivos (1 bloque(s) de ausencia)


Solo 3 de las 80 tiendas presentan días faltantes, con menos de 3 días de ausencia cada una — un porcentaje menor al 1% del periodo total. El hecho de que los días faltantes estén concentrados en bloques consecutivos sugiere problemas de conectividad o cierre temporal, y no errores de captura aleatorios.

Dado el volumen reducido, estas tiendas no serán excluidas del análisis. Sin embargo, durante el preprocesamiento será necesario definir si estos días se imputan o se excluyen del conjunto de entrenamiento para no introducir discontinuidades en las series temporales.

In [9]:
transactions.describe().round(2)

,total_transactions,cash_transactions,card_transactions,amount_total,amount_cash,amount_card,units_sold,avg_ticket,has_promotion,replenishment_signal
count,203958.00,191835.00,203958.00,203958.00,191835.00,203958.00,197840.00,199084.00,203958.0,202998.00
mean,613.51,273.98,313.10,170581.26,71623.69,98908.20,1175.41,411.76,0.2,614.45
std,631.77,323.69,286.56,150445.65,71687.13,88072.57,1264.96,303.64,0.4,538.10
min,14.00,3.00,11.00,6922.32,2721.28,4201.04,22.00,69.86,0.0,34.67
25%,228.00,77.00,136.00,78798.17,30519.27,44898.69,418.00,182.44,0.0,249.86
50%,424.00,169.00,234.00,129965.23,51462.28,74877.42,793.00,297.03,0.0,447.57
75%,783.00,352.00,399.00,213323.91,86819.46,124418.91,1487.00,534.14,0.0,809.57
max,15194.00,8387.00,6827.00,4050995.63,2294517.17,2269315.38,31936.00,1931.74,1.0,7064.00


In [11]:
transactions.store_id.nunique()

80

In [21]:
transactions.groupby(['category'])[['total_transactions']].sum().sort_values(by='total_transactions', ascending=False)

,total_transactions
category,
Abarrotes,40252500
Bebidas,31274705
Cuidado_Personal,20103737
Hogar,15623305
Ropa,11166847
Electronica,6709887


In [25]:
print(transactions["date"].min(), "to", transactions["date"].max())

2023-01-01 to 2024-02-29


In [52]:
transactions['has_promotion'].value_counts(normalize=True)

has_promotion
0    0.800228
1    0.199772
Name: proportion, dtype: float64

### Stores

In [31]:
stores.head(2)

,store_id,store_format,region,size_sqm,num_checkouts,opening_year,socioeconomic_level,has_pharmacy,has_fuel_station
0,STR_001,Supercenter,Norte,11072,37,2020,B,True,False
1,STR_002,Supercenter,Occidente,13502,22,2020,C+,True,False


In [33]:
stores.shape

(80, 9)

In [23]:
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   store_id             80 non-null     object
 1   store_format         80 non-null     object
 2   region               80 non-null     object
 3   size_sqm             80 non-null     int64 
 4   num_checkouts        80 non-null     int64 
 5   opening_year         80 non-null     int64 
 6   socioeconomic_level  80 non-null     object
 7   has_pharmacy         80 non-null     bool  
 8   has_fuel_station     80 non-null     bool  
dtypes: bool(2), int64(3), object(4)
memory usage: 4.7+ KB


In [32]:
stores.store_id.nunique()

80

In [26]:
stores.store_format.value_counts()

store_format
Bodega         35
Express        25
Supercenter    20
Name: count, dtype: int64

In [27]:
stores.region.value_counts()

region
Occidente    20
Oriente      18
Norte        17
Sur          13
Centro       12
Name: count, dtype: int64

In [30]:
stores.size_sqm.describe().round(2)

count       80.00
mean      5991.05
std       4018.84
min        737.00
25%       2637.50
50%       5135.00
75%       8047.50
max      14784.00
Name: size_sqm, dtype: float64

In [34]:
stores.socioeconomic_level.value_counts() 

socioeconomic_level
C+     27
C      27
B      21
A/B     5
Name: count, dtype: int64

In [35]:
stores.has_pharmacy.sum()

15

In [36]:
stores.has_fuel_station.sum()

22

### Calendar

In [37]:
calendar.head(2)

,date,day_of_week,day_name,week_of_year,month,year,quarter,season,is_holiday,holiday_name,is_payday,is_weekend,is_navidad_season,is_buen_fin,is_semana_santa
0,2023-01-01,6,Sunday,52,1,2023,1,Invierno,True,Año Nuevo,False,True,True,False,False
1,2023-01-02,0,Monday,1,1,2023,1,Invierno,False,NaN,False,False,True,False,False


In [39]:
calendar.shape

(425, 15)

In [40]:
print(calendar["date"].min(), "to", calendar["date"].max())

2023-01-01 to 2024-02-29


In [51]:
calendar['date'] = pd.to_datetime(calendar['date'])

event_cols = ['is_holiday', 'is_payday', 'is_weekend', 'is_buen_fin', 'is_navidad_season', 'is_semana_santa']
calendar_year = calendar[calendar['date'].dt.year == 2023]
event_summary = pd.DataFrame({
    'Días en el periodo': [calendar_year[col].sum() for col in event_cols],
    '% del periodo': [(calendar_year[col].sum() / len(calendar_year) * 100).round(1) for col in event_cols],
}, index=event_cols)

event_summary

,Días en el periodo,% del periodo
is_holiday,16,4.4
is_payday,24,6.6
is_weekend,105,28.8
is_buen_fin,4,1.1
is_navidad_season,23,6.3
is_semana_santa,2,0.5


Las variables de calendario representan eventos con frecuencias muy distintas. Mientras que fines de semana y quincenas aparecen repetidamente a lo largo del periodo, eventos como Buen Fin o Semana Santa únicamente cuentan con una ocurrencia. Esta diferencia deberá considerarse al interpretar la importancia que el modelo asigne a estas variables.

---

### Observaciones

- El dataset contiene aproximadamente **204 mil registros**, correspondientes a **80 tiendas**, **6 categorías** y un periodo de **425 días** (enero de 2023 a febrero de 2024).
- La variable objetivo (total_transactions) presenta un **100% de completitud**, por lo que puede utilizarse directamente para el modelado.
- Los valores faltantes se concentran en variables específicas y responden a distintos contextos de negocio, por lo que requerirán estrategias de tratamiento diferenciadas.
- Solo **3 tiendas** presentan días sin información (<1% del periodo), concentrados en bloques consecutivos, lo que sugiere interrupciones operativas más que errores aleatorios.
- Aproximadamente el **20% de los registros** corresponde a días con promociones activas, mientras que el calendario incorpora eventos relevantes del retail mexicano como Buen Fin, Navidad y Semana Santa.

### Implicaciones para el modelo

- La alta completitud de total_transactions permite utilizarla como variable objetivo sin necesidad de imputación.
- Las variables de calendario y promociones representan información disponible antes de realizar la predicción, por lo que constituyen candidatos naturales para el modelo de forecasting.
- Los valores faltantes deberán tratarse de manera diferenciada según el significado de cada variable, evitando aplicar una única estrategia de imputación.
- Los periodos sin información identificados en algunas tiendas serán analizados durante el preprocesamiento para determinar el tratamiento más adecuado.
- Debido a que el periodo observado comprende un único ciclo anual, los efectos asociados a eventos como Buen Fin, Navidad y Semana Santa deberán interpretarse con cautela.

### Riesgos identificados

- Los eventos anuales sólo aparecen una vez.
- Algunas variables contienen valores faltantes.
- Existen pequeños periodos sin información en tres tiendas.